In [5]:
from pathlib import Path
import pandas as pd


def load_etf_data(folder):

    folder = Path(folder)

    df_etf = {}

    # 遍历所有指标文件
    for file in folder.glob("*.xlsx"):

        factor_name = file.stem

        print("loading:", factor_name)

        df = pd.read_excel(file)

        # 第一列改为日期
        date_col = df.columns[0]

        df = df.rename(columns={date_col: "日期"})
        df["日期"] = pd.to_datetime(df["日期"])

        # 每个ETF
        for etf in df.columns[1:]:

            if etf not in df_etf:

                df_etf[etf] = pd.DataFrame(
                    index=df["日期"]
                )

            df_etf[etf][factor_name] = df[etf].values

    # 排序
    for etf in df_etf:

        df_etf[etf] = df_etf[etf].sort_index()
        df_etf[etf].index.name = "date"

    return df_etf

In [10]:
FILE_PATH = "data/datas/轮动池5"
df_etf = load_etf_data(FILE_PATH)
folder = Path(FILE_PATH)

pd.to_pickle(df_etf, folder / "etf_panel.pkl")
print("保存完成：", folder / "etf_panel.pkl")

loading: 成交笔数
loading: 最低价
loading: 前收盘价
loading: 涨跌
loading: 振幅
loading: 收盘价
loading: 换手率
loading: 均价
loading: 开盘价
loading: 最高价
loading: 成交量
loading: 涨跌幅
loading: 成交额
保存完成： data/datas/轮动池5/etf_panel.pkl


In [ ]:
import pandas as pd
from pathlib import Path

file_path = Path("data/datas/轮动池6/etf_panel.pkl")

etf_panel = pd.read_pickle(file_path)

加载完成
ETF数量： 18
ETF列表： ['沪深300ETF华泰柏瑞', '中证500ETF南方', '中证1000ETF华泰柏瑞', '科创50ETF华夏', '华安创业板50ETF', '红利ETF华泰柏瑞', '红利ETF工银', '纳指ETF国泰', '恒生ETF华夏', '恒生科技ETF易方达', '港股红利ETF博时', '黄金ETF华安', '豆粕ETF华夏', '有色金属ETF南方', '能源ETF汇添富', '国债ETF', '十年国债ETF国泰', '货币ETF华夏']


In [10]:
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

import pandas as pd
import importlib

# 1️⃣ reload modules
import data.indicators as ind
import data.data_loader as dl
import portfolio.portfolio as pf
import strategy.single_factor_strategy as st
import strategy.single_factor_full_strategy as st_full
import strategy.single_factor_full_stploss_strategy as st_stp
import strategy.single_factor_abs_stp_strategy as abs_stp
import engine.instant_engine as eng1
import engine.delay_engine as eng2
import engine.instant_full_engine as eng3
import engine.delay_full_engine as eng4
import analysis.plot_analysis as ana

importlib.reload(ind)
importlib.reload(dl)
importlib.reload(pf)
importlib.reload(st)
importlib.reload(eng1)
importlib.reload(eng2)
importlib.reload(eng3)
importlib.reload(eng4)
importlib.reload(st_full)
importlib.reload(st_stp)
importlib.reload(abs_stp)
importlib.reload(ana)

# 2️⃣ re-import ALL classes (关键！！)
from data.indicators import pct_change, trend_score, rsrs, rsrs_std, trend_score_avg_price
from data.indicators import momentum_open, momentum_close, momentum_avg, trend_score_riskoff
from data.indicators import sharpe_momentum
from data.data_loader import load_etf_data

from portfolio.portfolio import Portfolio

from strategy.single_factor_strategy import SingleFactorStrategy
from strategy.single_factor_full_strategy import SingleFactorFullStrategy
from strategy.single_factor_full_stploss_strategy import SingleFactorFullStpLossStrategy
from strategy.single_factor_abs_stp_strategy import SingleFactorAbsStpStrategy

from engine.instant_full_engine import InstantFullEngine
from engine.delay_full_engine import DelayFullEngine


factor_df = trend_score(etf_panel, window=10)
factor_df

,沪深300ETF华泰柏瑞,中证500ETF南方,中证1000ETF华泰柏瑞,科创50ETF华夏,华安创业板50ETF,红利ETF华泰柏瑞,红利ETF工银,纳指ETF国泰,恒生ETF华夏,恒生科技ETF易方达,港股红利ETF博时,黄金ETF华安,豆粕ETF华夏,有色金属ETF南方,能源ETF汇添富,国债ETF,十年国债ETF国泰,货币ETF华夏
date,,,,,,,,,,,,,,,,,,
2015-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-05-29,0.421335,-0.028049,-0.211968,0.227598,3.968633,-0.059082,-0.113960,16.326724,-3.503533e-01,0.007207,-0.535096,-0.162252,-0.156251,0.002995,-0.060005,0.0,0.051422,0.000157
2026-06-01,0.096346,-0.198845,-0.405146,-0.068450,2.049902,0.038450,-0.090786,31.133005,-3.030075e-01,0.071668,-0.268145,-0.122181,-0.152811,-0.000534,0.046143,0.0,0.074788,0.000008
2026-06-02,0.104862,-0.291857,-0.425580,-0.311357,2.011163,0.646900,-0.051584,27.835664,-1.389340e-02,0.741844,-0.067302,-0.030186,-0.118184,-0.000604,1.097742,0.0,0.124241,0.000053
